In [1]:
%%writefile C:\Users\tuf81\app.py

import streamlit as st
import sqlite3

st.set_page_config(
    page_title="The Caregiver Circle",
    page_icon="🌿",
    layout="centered"
)

DB_PATH = r"C:\Users\tuf81\discussion_board.db"

def get_connection():
    return sqlite3.connect(DB_PATH)


def get_discussions():
    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute("""
    SELECT id, title, post, created_at
    FROM discussions
    ORDER BY created_at DESC
    """)

    discussions = cursor.fetchall()
    conn.close()

    return discussions


def create_discussion(title, post):
    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute("""
    INSERT INTO discussions (title, post)
    VALUES (?, ?)
    """, (title, post))

    conn.commit()
    conn.close()


def get_comments(discussion_id):
    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute("""
    SELECT id, comment, created_at
    FROM comments
    WHERE discussion_id = ?
    ORDER BY created_at ASC
    """, (discussion_id,))

    comments = cursor.fetchall()
    conn.close()

    return comments


def add_comment(discussion_id, comment):
    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute("""
    INSERT INTO comments (discussion_id, comment)
    VALUES (?, ?)
    """, (discussion_id, comment))

    conn.commit()
    conn.close()


# COMMUNITY HOMEPAGE

st.title("🌿 The Caregiver Circle")

st.write(
    "A welcoming community for caregivers "
    "to connect, share experiences, and support one another."
)

st.divider()

st.header("Start a Discussion")

with st.form("new_discussion"):
    title = st.text_input("Discussion title")

    post = st.text_area(
        "What's on your mind?",
        height=150
    )

    submitted = st.form_submit_button("Publish Discussion")

    if submitted:
        if title.strip() and post.strip():
            create_discussion(title.strip(), post.strip())
            st.success("Your discussion has been published!")
            st.rerun()
        else:
            st.warning("Please enter a title and discussion.")

st.divider()

st.header("Community Discussions")

discussions = get_discussions()

if not discussions:
    st.info("Be the first to start a discussion!")

for discussion in discussions:

    discussion_id = discussion[0]

    with st.container(border=True):

        st.subheader(discussion[1])
        st.write(discussion[2])

        st.caption(f"Posted: {discussion[3]}")

        st.write("**Comments**")

        comments = get_comments(discussion_id)

        if comments:
            for comment in comments:
                st.write(comment[1])
                st.caption(f"Posted: {comment[2]}")
        else:
            st.caption("No comments yet.")

        with st.form(f"comment_form_{discussion_id}"):

            new_comment = st.text_area(
                "Add a comment",
                key=f"comment_{discussion_id}"
            )

            comment_submitted = st.form_submit_button("Post Comment")

            if comment_submitted:
                if new_comment.strip():
                    add_comment(discussion_id, new_comment.strip())
                    st.success("Comment posted!")
                    st.rerun()
                else:
                    st.warning("Please enter a comment.")

Overwriting C:\Users\tuf81\app.py


In [2]:
import subprocess

subprocess.Popen(
    'start cmd /k "python -m streamlit run C:\\Users\\tuf81\\app.py"',
    shell=True
)

<Popen: returncode: None args: 'start cmd /k "python -m streamlit run C:\\Us...>

In [3]:
import os

os.makedirs(r"C:\Users\tuf81\pages", exist_ok=True)

print("Pages folder ready!")

Pages folder ready!


In [4]:
%%writefile C:\Users\tuf81\pages\1_Account.py

import streamlit as st
import sqlite3
import hashlib
import os
import hmac

st.set_page_config(
    page_title="The Caregiver Circle | Account",
    page_icon="🌿"
)

DB_PATH = r"C:\Users\tuf81\discussion_board.db"


def get_connection():
    return sqlite3.connect(DB_PATH)


def hash_password(password):
    salt = os.urandom(16)

    password_hash = hashlib.pbkdf2_hmac(
        "sha256",
        password.encode(),
        salt,
        600000
    )

    return salt.hex() + ":" + password_hash.hex()


def verify_password(password, stored_hash):
    salt, original_hash = stored_hash.split(":")

    new_hash = hashlib.pbkdf2_hmac(
        "sha256",
        password.encode(),
        bytes.fromhex(salt),
        600000
    )

    return hmac.compare_digest(
        new_hash.hex(),
        original_hash
    )


def register_user(username, email, password):
    conn = get_connection()
    cursor = conn.cursor()

    try:
        password_hash = hash_password(password)

        cursor.execute("""
        INSERT INTO users (username, email, password_hash)
        VALUES (?, ?, ?)
        """, (username, email, password_hash))

        conn.commit()
        return True, "Account created successfully!"

    except sqlite3.IntegrityError:
        return False, "Username or email already exists."

    finally:
        conn.close()


def login_user(username, password):
    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute("""
    SELECT id, password_hash
    FROM users
    WHERE username = ?
    """, (username,))

    result = cursor.fetchone()
    conn.close()

    if result is None:
        return False, None

    if verify_password(password, result[1]):
        return True, result[0]

    return False, None


# ACCOUNT PAGE

st.title("🌿 The Caregiver Circle")
st.header("Your Account")

if "user_id" not in st.session_state:
    st.session_state.user_id = None

if "username" not in st.session_state:
    st.session_state.username = None


if st.session_state.user_id is None:

    create_tab, login_tab = st.tabs(
        ["Create Account", "Login"]
    )

    with create_tab:

        st.subheader("Join the Community")

        with st.form("registration_form"):

            username = st.text_input("Username")
            email = st.text_input("Email")
            password = st.text_input(
                "Password",
                type="password"
            )
            confirm_password = st.text_input(
                "Confirm Password",
                type="password"
            )

            register_button = st.form_submit_button(
                "Create Account"
            )

            if register_button:

                if not username.strip() or not email.strip() or not password:
                    st.warning("Please complete all fields.")

                elif len(password) < 8:
                    st.warning("Password must be at least 12 characters.")

                elif password != confirm_password:
                    st.warning("Passwords do not match.")

                else:
                    success, message = register_user(
                        username.strip(),
                        email.strip().lower(),
                        password
                    )

                    if success:
                        st.success(message)
                    else:
                        st.error(message)

    with login_tab:

        st.subheader("Welcome Back")

        with st.form("login_form"):

            login_username = st.text_input("Username")
            login_password = st.text_input(
                "Password",
                type="password"
            )

            login_button = st.form_submit_button("Login")

            if login_button:

                success, user_id = login_user(
                    login_username.strip(),
                    login_password
                )

                if success:
                    st.session_state.user_id = user_id
                    st.session_state.username = login_username.strip()
                    st.rerun()
                else:
                    st.error("Incorrect username or password.")

else:

    st.success(
        f"Welcome back, {st.session_state.username}!"
    )

    if st.button("Log Out"):
        st.session_state.user_id = None
        st.session_state.username = None
        st.rerun()

Overwriting C:\Users\tuf81\pages\1_Account.py


In [5]:
import subprocess

subprocess.Popen(
    'cmd /k "cd /d C:\\Users\\tuf81 && python -m streamlit run app.py"',
    creationflags=subprocess.CREATE_NEW_CONSOLE
)

<Popen: returncode: None args: 'cmd /k "cd /d C:\\Users\\tuf81 && python -m ...>

In [6]:
code = '''
import streamlit as st
import sqlite3

st.set_page_config(
    page_title="My Wellness | The Caregiver Circle",
    page_icon="🌿"
)

st.title("🌿 My Wellness")

st.write(
    "Your personal space to reflect, track your wellness, "
    "and take care of yourself."
)

st.info("Your personal wellness information is private.")

st.header("Your Wellness Space")

st.write("Coming soon:")
st.write("- Personal wellness check-ins")
st.write("- Private diary")
st.write("- Physical health tracking")
'''

with open(r"C:\Users\tuf81\pages\2_My_Wellness.py", "w", encoding="utf-8") as file:
    file.write(code)

print("My Wellness page created!")

My Wellness page created!


In [7]:
## Check the database
import sqlite3

conn = sqlite3.connect(
    r"C:\Users\tuf81\discussion_board.db"
)

cursor = conn.cursor()

cursor.execute("""
SELECT name FROM sqlite_master
WHERE type = 'table'
""")

for table in cursor.fetchall():
    print(table[0])

conn.close()

discussions
sqlite_sequence
comments
users
wellness_checkins
diary_entries
physical_health


In [8]:
## Create the wellness tables 

import sqlite3

conn = sqlite3.connect(
    r"C:\Users\tuf81\discussion_board.db"
)

cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS wellness_checkins (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id INTEGER NOT NULL,
    mood TEXT,
    stress_level INTEGER,
    physical_wellness TEXT,
    notes TEXT,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS diary_entries (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id INTEGER NOT NULL,
    title TEXT,
    entry TEXT NOT NULL,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS physical_health (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id INTEGER NOT NULL,
    sleep_hours REAL,
    activity_minutes INTEGER,
    energy_level INTEGER,
    notes TEXT,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
""")

conn.commit()
conn.close()

print("Private wellness tables created!")

Private wellness tables created!


In [9]:
## Verify the tables 

import sqlite3

conn = sqlite3.connect(
    r"C:\Users\tuf81\discussion_board.db"
)

cursor = conn.cursor()

cursor.execute("""
SELECT name FROM sqlite_master
WHERE type = 'table'
""")

for table in cursor.fetchall():
    print(table[0])

conn.close()

discussions
sqlite_sequence
comments
users
wellness_checkins
diary_entries
physical_health


In [10]:
## Build the dashboard 

code = '''
import streamlit as st
import sqlite3

st.set_page_config(
    page_title="My Wellness | The Caregiver Circle",
    page_icon="🌿"
)

DB_PATH = r"C:\\Users\\tuf81\\discussion_board.db"

def get_connection():
    return sqlite3.connect(DB_PATH)

st.title("🌿 My Wellness")

if "user_id" not in st.session_state or st.session_state.user_id is None:

    st.warning("Please log in to access your private wellness space.")

    st.page_link("pages/1_Account.py", label="Go to Account")

else:

    st.write(f"Welcome, {st.session_state.username}!")

    st.info("Your personal wellness information is private.")

    st.header("Your Wellness Space")

    st.write("Your personal tools will appear here.")
'''

with open(r"C:\Users\tuf81\pages\2_My_Wellness.py", "w", encoding="utf-8") as file:
    file.write(code)

print("Private wellness dashboard updated!")

Private wellness dashboard updated!


In [11]:
## Restart the app 

import subprocess

subprocess.Popen(
    'cmd /k "cd /d C:\\Users\\tuf81 && python -m streamlit run app.py"',
    creationflags=subprocess.CREATE_NEW_CONSOLE
)

<Popen: returncode: None args: 'cmd /k "cd /d C:\\Users\\tuf81 && python -m ...>

In [12]:
account_path = r"C:\Users\tuf81\pages\1_Account.py"

with open(account_path, "r", encoding="utf-8") as file:
    code = file.read()

# Store new usernames in lowercase
code = code.replace(
    '""", (username, email, password_hash))',
    '""", (username.strip().lower(), email, password_hash))'
)

# Make login ignore capitalization
code = code.replace(
    'WHERE username = ?',
    'WHERE username = ? COLLATE NOCASE'
)

with open(account_path, "w", encoding="utf-8") as file:
    file.write(code)

print("Username case-insensitivity updated!")

Username case-insensitivity updated!


In [13]:
## Create a private diary 

code = r'''
import streamlit as st
import sqlite3

st.set_page_config(
    page_title="My Wellness | The Caregiver Circle",
    page_icon="🌿"
)

DB_PATH = r"C:\Users\tuf81\discussion_board.db"

def get_connection():
    return sqlite3.connect(DB_PATH)

def save_diary_entry(user_id, title, entry):
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("""
        INSERT INTO diary_entries (user_id, title, entry)
        VALUES (?, ?, ?)
    """, (user_id, title, entry))
    conn.commit()
    conn.close()

def get_diary_entries(user_id):
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("""
        SELECT id, title, entry, created_at
        FROM diary_entries
        WHERE user_id = ?
        ORDER BY created_at DESC
    """, (user_id,))
    entries = cursor.fetchall()
    conn.close()
    return entries

def update_diary_entry(entry_id, user_id, title, entry):
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("""
        UPDATE diary_entries
        SET title = ?, entry = ?
        WHERE id = ? AND user_id = ?
    """, (title, entry, entry_id, user_id))
    conn.commit()
    conn.close()

def delete_diary_entry(entry_id, user_id):
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("""
        DELETE FROM diary_entries
        WHERE id = ? AND user_id = ?
    """, (entry_id, user_id))
    conn.commit()
    conn.close()

st.title("🌿 My Wellness")

if "user_id" not in st.session_state or st.session_state.user_id is None:
    st.warning("Please log in to access your private wellness space.")
    st.page_link("pages/1_Account.py", label="Go to Account")

else:
    user_id = st.session_state.user_id

    st.write(f"Welcome, {st.session_state.username}!")
    st.caption("Your personal wellness space is private.")

    st.divider()
    st.header("📖 My Private Diary")
    st.write("A space to reflect, write, and keep track of your experiences.")

    with st.form("new_diary_entry"):
        title = st.text_input("Entry title")
        entry = st.text_area(
            "What's on your mind?",
            height=200
        )
        submitted = st.form_submit_button("Save Diary Entry")

        if submitted:
            if entry.strip():
                save_diary_entry(
                    user_id,
                    title.strip(),
                    entry.strip()
                )
                st.success("Your diary entry has been saved.")
                st.rerun()
            else:
                st.warning("Please write something before saving.")

    st.divider()
    st.subheader("Previous Entries")

    entries = get_diary_entries(user_id)

    if not entries:
        st.info("Your diary is empty. Your first entry can start here.")

    for diary in entries:
        entry_id, entry_title, entry_text, created_at = diary

        with st.expander(
            f"{entry_title or 'Untitled Entry'} — {created_at}"
        ):
            st.write(entry_text)

            st.divider()
            st.write("**Edit this entry**")

            with st.form(f"edit_form_{entry_id}"):
                edited_title = st.text_input(
                    "Title",
                    value=entry_title or "",
                    key=f"title_{entry_id}"
                )
                edited_text = st.text_area(
                    "Entry",
                    value=entry_text,
                    height=150,
                    key=f"text_{entry_id}"
                )

                update_button = st.form_submit_button("Save Changes")

                if update_button:
                    if edited_text.strip():
                        update_diary_entry(
                            entry_id,
                            user_id,
                            edited_title.strip(),
                            edited_text.strip()
                        )
                        st.success("Entry updated.")
                        st.rerun()
                    else:
                        st.warning("An entry cannot be empty.")

            if st.button("Delete Entry", key=f"delete_{entry_id}"):
                delete_diary_entry(entry_id, user_id)
                st.success("Entry deleted.")
                st.rerun()
'''

with open(
    r"C:\Users\tuf81\pages\2_My_Wellness.py",
    "w",
    encoding="utf-8"
) as file:
    file.write(code)

print("Private Diary page created successfully!")


Private Diary page created successfully!


In [14]:
## Run it!
import subprocess

subprocess.Popen(
    'cmd /k "cd /d C:\\Users\\tuf81 && python -m streamlit run app.py"',
    creationflags=subprocess.CREATE_NEW_CONSOLE
)

<Popen: returncode: None args: 'cmd /k "cd /d C:\\Users\\tuf81 && python -m ...>

In [15]:
code = r'''
import streamlit as st
import sqlite3

st.set_page_config(
    page_title="My Wellness | The Caregiver Circle",
    page_icon="🌿"
)

DB_PATH = r"C:\Users\tuf81\discussion_board.db"

def get_connection():
    return sqlite3.connect(DB_PATH)

# PRIVATE DIARY FUNCTIONS

def save_diary_entry(user_id, title, entry):
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("""
        INSERT INTO diary_entries (user_id, title, entry)
        VALUES (?, ?, ?)
    """, (user_id, title, entry))
    conn.commit()
    conn.close()

def get_diary_entries(user_id):
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("""
        SELECT id, title, entry, created_at
        FROM diary_entries
        WHERE user_id = ?
        ORDER BY created_at DESC
    """, (user_id,))
    entries = cursor.fetchall()
    conn.close()
    return entries

def update_diary_entry(entry_id, user_id, title, entry):
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("""
        UPDATE diary_entries
        SET title = ?, entry = ?
        WHERE id = ? AND user_id = ?
    """, (title, entry, entry_id, user_id))
    conn.commit()
    conn.close()

def delete_diary_entry(entry_id, user_id):
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("""
        DELETE FROM diary_entries
        WHERE id = ? AND user_id = ?
    """, (entry_id, user_id))
    conn.commit()
    conn.close()

# WELLNESS CHECK-IN FUNCTIONS

def save_checkin(user_id, mood, stress, physical_wellness, notes):
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("""
        INSERT INTO wellness_checkins
        (user_id, mood, stress_level, physical_wellness, notes)
        VALUES (?, ?, ?, ?, ?)
    """, (user_id, mood, stress, physical_wellness, notes))
    conn.commit()
    conn.close()

def get_checkins(user_id):
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("""
        SELECT mood, stress_level, physical_wellness, notes, created_at
        FROM wellness_checkins
        WHERE user_id = ?
        ORDER BY created_at DESC
    """, (user_id,))
    checkins = cursor.fetchall()
    conn.close()
    return checkins

# PAGE DISPLAY

st.title("🌿 My Wellness")

if "user_id" not in st.session_state or st.session_state.user_id is None:
    st.warning("Please log in to access your private wellness space.")
    st.page_link("pages/1_Account.py", label="Go to Account")

else:
    user_id = st.session_state.user_id

    st.write(f"Welcome, {st.session_state.username}!")
    st.caption("Your personal wellness space is private.")

    # WELLNESS CHECK-IN

    st.divider()
    st.header("💚 Wellness Check-In")
    st.write("Take a moment to check in with yourself.")

    with st.form("wellness_checkin_form"):

        mood = st.selectbox(
            "How are you feeling today?",
            [
                "Very good",
                "Good",
                "Okay",
                "Not great",
                "Struggling"
            ]
        )

        stress = st.slider(
            "How would you rate your stress level?",
            min_value=1,
            max_value=10,
            value=5,
            help="1 = Very low stress | 10 = Extremely high stress"
        )

        physical_wellness = st.selectbox(
            "How are you feeling physically?",
            [
                "Great",
                "Good",
                "Okay",
                "Tired",
                "Not feeling well"
            ]
        )

        notes = st.text_area(
            "Anything else you'd like to share? (Optional)",
            height=100
        )

        submitted = st.form_submit_button("Save Check-In")

        if submitted:
            save_checkin(
                user_id,
                mood,
                stress,
                physical_wellness,
                notes.strip()
            )
            st.success("Your wellness check-in has been saved!")
            st.rerun()

    # CHECK-IN HISTORY

    st.subheader("Previous Check-Ins")

    checkins = get_checkins(user_id)

    if not checkins:
        st.info("Your check-in history will appear here.")

    for checkin in checkins:
        mood, stress, physical, notes, created_at = checkin

        with st.expander(f"Check-In — {created_at}"):
            st.write(f"**Mood:** {mood}")
            st.write(f"**Stress level:** {stress}/10")
            st.write(f"**Physical wellness:** {physical}")

            if notes:
                st.write(f"**Notes:** {notes}")

    # PRIVATE DIARY

    st.divider()
    st.header("📖 My Private Diary")
    st.write("A private space to reflect, write, and revisit your experiences.")

    with st.form("new_diary_entry"):
        title = st.text_input("Entry title")
        entry = st.text_area("What's on your mind?", height=200)

        submitted = st.form_submit_button("Save Diary Entry")

        if submitted:
            if entry.strip():
                save_diary_entry(user_id, title.strip(), entry.strip())
                st.success("Your diary entry has been saved.")
                st.rerun()
            else:
                st.warning("Please write something before saving.")

    st.subheader("Previous Diary Entries")

    entries = get_diary_entries(user_id)

    if not entries:
        st.info("Your diary is empty. Your first entry can start here.")

    for diary in entries:
        entry_id, entry_title, entry_text, created_at = diary

        with st.expander(
            f"{entry_title or 'Untitled Entry'} — {created_at}"
        ):
            st.write(entry_text)
            st.divider()
            st.write("**Edit this entry**")

            with st.form(f"edit_form_{entry_id}"):
                edited_title = st.text_input(
                    "Title",
                    value=entry_title or "",
                    key=f"title_{entry_id}"
                )

                edited_text = st.text_area(
                    "Entry",
                    value=entry_text,
                    height=150,
                    key=f"text_{entry_id}"
                )

                update_button = st.form_submit_button("Save Changes")

                if update_button:
                    if edited_text.strip():
                        update_diary_entry(
                            entry_id,
                            user_id,
                            edited_title.strip(),
                            edited_text.strip()
                        )
                        st.success("Entry updated.")
                        st.rerun()
                    else:
                        st.warning("An entry cannot be empty.")

            if st.button("Delete Entry", key=f"delete_{entry_id}"):
                delete_diary_entry(entry_id, user_id)
                st.success("Entry deleted.")
                st.rerun()
'''

with open(
    r"C:\Users\tuf81\pages\2_My_Wellness.py",
    "w",
    encoding="utf-8"
) as file:
    file.write(code)

print("Wellness Check-In and Private Diary updated successfully!")


Wellness Check-In and Private Diary updated successfully!


In [16]:
## Run it!

import subprocess

subprocess.Popen(
    'cmd /k "cd /d C:\\Users\\tuf81 && python -m streamlit run app.py"',
    creationflags=subprocess.CREATE_NEW_CONSOLE
)

<Popen: returncode: None args: 'cmd /k "cd /d C:\\Users\\tuf81 && python -m ...>

In [17]:
## Physical Wellness Tracker

code = r'''
import streamlit as st
import sqlite3

st.set_page_config(
    page_title="My Wellness | The Caregiver Circle",
    page_icon="🌿"
)

DB_PATH = r"C:\Users\tuf81\discussion_board.db"

def get_connection():
    return sqlite3.connect(DB_PATH)

# PRIVATE DIARY FUNCTIONS

def save_diary_entry(user_id, title, entry):
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("""
        INSERT INTO diary_entries (user_id, title, entry)
        VALUES (?, ?, ?)
    """, (user_id, title, entry))
    conn.commit()
    conn.close()

def get_diary_entries(user_id):
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("""
        SELECT id, title, entry, created_at
        FROM diary_entries
        WHERE user_id = ?
        ORDER BY created_at DESC
    """, (user_id,))
    entries = cursor.fetchall()
    conn.close()
    return entries

def update_diary_entry(entry_id, user_id, title, entry):
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("""
        UPDATE diary_entries
        SET title = ?, entry = ?
        WHERE id = ? AND user_id = ?
    """, (title, entry, entry_id, user_id))
    conn.commit()
    conn.close()

def delete_diary_entry(entry_id, user_id):
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("""
        DELETE FROM diary_entries
        WHERE id = ? AND user_id = ?
    """, (entry_id, user_id))
    conn.commit()
    conn.close()

# WELLNESS CHECK-IN FUNCTIONS

def save_checkin(user_id, mood, stress, physical_wellness, notes):
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("""
        INSERT INTO wellness_checkins
        (user_id, mood, stress_level, physical_wellness, notes)
        VALUES (?, ?, ?, ?, ?)
    """, (user_id, mood, stress, physical_wellness, notes))
    conn.commit()
    conn.close()

def get_checkins(user_id):
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("""
        SELECT mood, stress_level, physical_wellness, notes, created_at
        FROM wellness_checkins
        WHERE user_id = ?
        ORDER BY created_at DESC
    """, (user_id,))
    checkins = cursor.fetchall()
    conn.close()
    return checkins

# PHYSICAL HEALTH FUNCTIONS

def save_physical_health(user_id, sleep, activity, energy, notes):
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("""
        INSERT INTO physical_health
        (user_id, sleep_hours, activity_minutes, energy_level, notes)
        VALUES (?, ?, ?, ?, ?)
    """, (user_id, sleep, activity, energy, notes))
    conn.commit()
    conn.close()

def get_physical_health(user_id):
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("""
        SELECT sleep_hours, activity_minutes, energy_level, notes, created_at
        FROM physical_health
        WHERE user_id = ?
        ORDER BY created_at DESC
    """, (user_id,))
    records = cursor.fetchall()
    conn.close()
    return records

# PAGE DISPLAY

st.title("🌿 My Wellness")

if "user_id" not in st.session_state or st.session_state.user_id is None:
    st.warning("Please log in to access your private wellness space.")
    st.page_link("pages/1_Account.py", label="Go to Account")

else:
    user_id = st.session_state.user_id

    st.write(f"Welcome, {st.session_state.username}!")
    st.caption("Your personal wellness space is private.")

    # WELLNESS CHECK-IN

    st.divider()
    st.header("💚 Wellness Check-In")
    st.write("Take a moment to check in with yourself.")

    with st.form("wellness_checkin_form"):
        mood = st.selectbox(
            "How are you feeling today?",
            ["Very good", "Good", "Okay", "Not great", "Struggling"]
        )

        stress = st.slider(
            "How would you rate your stress level?",
            min_value=1,
            max_value=10,
            value=5,
            help="1 = Very low stress | 10 = Extremely high stress"
        )

        physical_wellness = st.selectbox(
            "How are you feeling physically?",
            ["Great", "Good", "Okay", "Tired", "Not feeling well"]
        )

        notes = st.text_area(
            "Anything else you'd like to share? (Optional)",
            height=100
        )

        submitted = st.form_submit_button("Save Check-In")

        if submitted:
            save_checkin(
                user_id, mood, stress, physical_wellness, notes.strip()
            )
            st.success("Your wellness check-in has been saved!")
            st.rerun()

    st.subheader("Previous Check-Ins")
    checkins = get_checkins(user_id)

    if not checkins:
        st.info("Your check-in history will appear here.")

    for checkin in checkins:
        mood, stress, physical, notes, created_at = checkin

        with st.expander(f"Check-In — {created_at}"):
            st.write(f"**Mood:** {mood}")
            st.write(f"**Stress level:** {stress}/10")
            st.write(f"**Physical wellness:** {physical}")
            if notes:
                st.write(f"**Notes:** {notes}")

    # PHYSICAL HEALTH TRACKER

    st.divider()
    st.header("🏃 Physical Health Tracker")
    st.write("Keep track of your physical wellness over time.")

    with st.form("physical_health_form"):
        sleep = st.number_input(
            "How many hours did you sleep?",
            min_value=0.0,
            max_value=24.0,
            value=7.0,
            step=0.5
        )

        activity = st.number_input(
            "Physical activity (minutes)",
            min_value=0,
            max_value=1440,
            value=0,
            step=5
        )

        energy = st.slider(
            "How would you rate your energy level?",
            min_value=1,
            max_value=10,
            value=5,
            help="1 = Very low energy | 10 = Very high energy"
        )

        health_notes = st.text_area(
            "Additional physical health notes (Optional)",
            height=100
        )

        health_submitted = st.form_submit_button("Save Physical Health Record")

        if health_submitted:
            save_physical_health(
                user_id,
                sleep,
                activity,
                energy,
                health_notes.strip()
            )
            st.success("Your physical health record has been saved!")
            st.rerun()

    st.subheader("Physical Health History")
    health_records = get_physical_health(user_id)

    if not health_records:
        st.info("Your physical health history will appear here.")

    for record in health_records:
        sleep, activity, energy, notes, created_at = record

        with st.expander(f"Physical Health — {created_at}"):
            st.write(f"**Sleep:** {sleep} hours")
            st.write(f"**Activity:** {activity} minutes")
            st.write(f"**Energy:** {energy}/10")
            if notes:
                st.write(f"**Notes:** {notes}")

    # PRIVATE DIARY

    st.divider()
    st.header("📖 My Private Diary")
    st.write("A private space to reflect, write, and revisit your experiences.")

    with st.form("new_diary_entry"):
        title = st.text_input("Entry title")
        entry = st.text_area("What's on your mind?", height=200)

        submitted = st.form_submit_button("Save Diary Entry")

        if submitted:
            if entry.strip():
                save_diary_entry(user_id, title.strip(), entry.strip())
                st.success("Your diary entry has been saved.")
                st.rerun()
            else:
                st.warning("Please write something before saving.")

    st.subheader("Previous Diary Entries")
    entries = get_diary_entries(user_id)

    if not entries:
        st.info("Your diary is empty. Your first entry can start here.")

    for diary in entries:
        entry_id, entry_title, entry_text, created_at = diary

        with st.expander(
            f"{entry_title or 'Untitled Entry'} — {created_at}"
        ):
            st.write(entry_text)
            st.divider()
            st.write("**Edit this entry**")

            with st.form(f"edit_form_{entry_id}"):
                edited_title = st.text_input(
                    "Title",
                    value=entry_title or "",
                    key=f"title_{entry_id}"
                )

                edited_text = st.text_area(
                    "Entry",
                    value=entry_text,
                    height=150,
                    key=f"text_{entry_id}"
                )

                update_button = st.form_submit_button("Save Changes")

                if update_button:
                    if edited_text.strip():
                        update_diary_entry(
                            entry_id, user_id,
                            edited_title.strip(), edited_text.strip()
                        )
                        st.success("Entry updated.")
                        st.rerun()
                    else:
                        st.warning("An entry cannot be empty.")

            if st.button("Delete Entry", key=f"delete_{entry_id}"):
                delete_diary_entry(entry_id, user_id)
                st.success("Entry deleted.")
                st.rerun()
'''

with open(
    r"C:\Users\tuf81\pages\2_My_Wellness.py",
    "w",
    encoding="utf-8"
) as file:
    file.write(code)

print("Physical Health Tracker added successfully!")


Physical Health Tracker added successfully!


In [18]:
## Run that bad boy 

import subprocess

subprocess.Popen(
    'cmd /k "cd /d C:\\Users\\tuf81 && python -m streamlit run app.py"',
    creationflags=subprocess.CREATE_NEW_CONSOLE
)

<Popen: returncode: None args: 'cmd /k "cd /d C:\\Users\\tuf81 && python -m ...>